In [5]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.metrics import (
    precision_recall_curve, auc, cohen_kappa_score, f1_score,
    recall_score, precision_score, accuracy_score, confusion_matrix,
    roc_auc_score, balanced_accuracy_score, matthews_corrcoef,
    log_loss, brier_score_loss, fbeta_score
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
import matplotlib.pyplot as plt
import seaborn as sns
import time
from pathlib import Path
import logging
import os
import sys
import traceback
import importlib.util
from typing import Dict, Tuple, List, Any, Optional
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from google.colab import drive
import warnings

# Logging system configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('/tmp/pfl_lightgbm.log')
    ]
)
logger = logging.getLogger(__name__)

# Definition of the PFLZ_CP class as a simple placeholder/interface.
# Its methods will be provided by the PFL module loaded dynamically.

class PFLZ_CP:
    def __init__(self, n_clusters: int, epsilon: float = 1e-8, lgb_params: Optional[Dict[str, Any]] = None, feature_names: Optional[List[str]] = None) -> None:
        self.n_clusters = n_clusters
        self.epsilon = epsilon
        self.lgb_params = lgb_params if lgb_params is not None else {}
        self.feature_names = feature_names
        self.cluster_centers_ = np.array([])
        self.cluster_model = None
        self.cond_probs_ = None
        self.n_original_features_ = 0
        self.n_fuzzy_features_ = n_clusters
        self.X_original_fit = None

    # All methods like calc_mf, fit, predict_proba, etc. ARE DEFINED IN PFL.
    # They are expected from the PFL module during dynamic loading.
    # If the PFL.py module is not loaded correctly, calls to these methods
    # on a PFL_CP instance will raise an AttributeError, which is the desired behavior

# Global variable to track if the real PFL module has been successfully loaded
_PFLZ_MODULE_LOADED = False

# --- Module 1: Centralized Configuration ---

class ConfigurationSysteme:
    """Centralized configuration with validation and fixed clinical thresholds."""

    # Fixed system parameters
    NB_FEATURES = None
    NB_REGLES = 7
    INDICES_FEATURES = None
    EPSILON = 1e-8

    # Fixed clinical thresholds
    SEUILS_RISQUE_CLINIQUES = {
        'FAIBLE': 10.0,
        'MOYEN': 15.0,
        'ELEVE': 25.0
    }

    # LightGBM hyperparameters for consistency
    PARAMS_LGB = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'learning_rate': 0.08,
        'num_leaves': 25,
        'max_depth': 4,
        'min_data_in_leaf': 10,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.85,
        'bagging_freq': 5,
        'lambda_l1': 0.3,
        'lambda_l2': 0.6,
        'min_gain_to_split': 0.02,
        'path_smooth': 100,
        'verbose': -1,
        'random_state': 42,
        'force_col_wise': True,
        'scale_pos_weight': 1
    }

    # Interpretability parameters
    LAMBDA_COHERENCE = 0.15
    SEUIL_COHERENCE_ALERTE = 0.10

    PERCENTILE_COVERAGE = 60

    # Calibration parameters
    CALIBRATION_METHOD = 'isotonic'
    CALIBRATION_CV = 3

    # Store scalers for denormalization
    SCALERS = {}

    @classmethod
    def valider_configuration(cls):
        """Validates configuration consistency."""
        if cls.NB_FEATURES is not None and cls.INDICES_FEATURES is not None:
            if len(cls.INDICES_FEATURES) != cls.NB_FEATURES:
                raise ValueError("Inconsistency between NB_FEATURES and INDICES_FEATURES")
        logger.info("✅ Configuration validated")

# --- Module 2: Data Management ---

class GestionnaireDonnees:
    """Manages Drive mounting, location and data loading."""
    def __init__(self):
        self.mydrive_path = None
        self.modeling_data2_path = None
        self.all_feature_names = None
        # Attributes to store loaded DataFrames for easy access
        self.df_train = None
        self.X_train_df = None
        self.y_train_arr = None

    def monter_drive(self):
        """Mounts Drive in the file system."""
        print("="*50)
        print("DRIVE MOUNTING")
        print("="*50)
        drive.mount('/content/drive', force_remount=True)
        self.mydrive_path = Path("/content/drive/MyDrive")
        print("Drive mounted successfully")

    def localiser_dossier_donnees(self):
        """Locates the 'modeling_data2' folder in Drive."""
        print("\n" + "="*50)
        print("DATA FOLDER LOCATION")
        print("="*50)
        # Direct search for 'modeling_data2' folder under 'PFL'
        pfl_dir_candidate = self.mydrive_path / "PFL"
        if pfl_dir_candidate.is_dir() and (pfl_dir_candidate / "modeling_data2").is_dir():
            self.modeling_data2_path = pfl_dir_candidate / "modeling_data2"
            print(f"✅ modeling_data2 folder found: {self.modeling_data2_path}")
            return

        # Fallback: general search if direct path fails
        for root, dirs, _ in os.walk(self.mydrive_path):
            if "modeling_data2" in dirs:
                self.modeling_data2_path = Path(root) / "modeling_data2"
                print(f"✅ modeling_data2 folder found: {self.modeling_data2_path}")
                return
        raise FileNotFoundError("modeling_data2 folder not found in Drive")

    def charger_donnees(self):
        """Loads training data (df_train, X_train, y_train) from the folder."""
        print("\n" + "="*50)
        print("DATA LOADING")
        print("="*50)
        df_train_path = self.modeling_data2_path / 'df_train.hdf'
        if not df_train_path.exists():
            raise FileNotFoundError(f"File not found: {df_train_path}")
        self.df_train = pd.read_hdf(df_train_path, key='Processed_data')

        X_train_df_temp = pd.read_hdf(self.modeling_data2_path / 'X_train.hdf', key='Processed_data')
        self.all_feature_names = [col for col in X_train_df_temp.columns if col != 'A']

        A_train = self.df_train['A'].values
        self.X_train_df = pd.read_hdf(self.modeling_data2_path / 'X_train.hdf', key='Processed_data')
        self.X_train_df['A'] = A_train

        self.y_train_arr = pd.read_hdf(self.modeling_data2_path / 'y_train.hdf', key='Processed_data').values.ravel()

        # --- Calculation and storage of true min/max values ---
        # print("\n" + "="*50)
        # print("CALCULATION AND STORAGE OF REAL MIN/MAX VALUES FOR NORMALIZATION")
        # print("="*50)

        real_min_max_values = {}
        # Iterate over self.all_feature_names which already contains all feature names except 'A'
        for feature_name in self.all_feature_names:
            if feature_name in self.X_train_df.columns and pd.api.types.is_numeric_dtype(self.X_train_df[feature_name]):
                min_val = self.X_train_df[feature_name].min()
                max_val = self.X_train_df[feature_name].max()
                real_min_max_values[feature_name] = (min_val, max_val)
            # else:
                # print(f"Warning: Feature '{feature_name}' is not numeric or was not found in X_train_df for min/max calculation.")


        ConfigurationSysteme.SCALERS['min_max'] = real_min_max_values

        print("\n🌟 Data loaded successfully!")
        return self.X_train_df, self.y_train_arr, self.all_feature_names

# --- Module 3: PFL Module Loading ---

class ChargeurModulePFLZ:
    """Manages dynamic loading of the PFLZ class (alias PFLZ_CP) from its external file
    ensuring it is properly imported."""
    @staticmethod
    def charger_module_pflz_cp(mydrive_path: Path):
        print("\n" + "="*50)
        print("PFL MODULE LOADING")
        print("="*50)
        pflz_file_path = None
        # Look for PFLZ.py file in the Modules directory
        candidate_path = mydrive_path / "PFL" / "Modules" / "PFLZ.py"
        if candidate_path.exists():
            pflz_file_path = candidate_path
        else:
            # Fallback: broader search if direct path doesn't work
            for root, dirs, files in os.walk(mydrive_path):
                if "PFL" in dirs:
                    pfl_dir = Path(root) / "PFL"
                    if "Modules" in os.listdir(pfl_dir):
                        modules_path = pfl_dir / "Modules"
                        if "PFLZ.py" in os.listdir(modules_path):
                            pflz_file_path = modules_path / "PFLZ.py"
                            break

        if pflz_file_path is None:
            raise FileNotFoundError("PFLZ.py file not found in Drive. Please ensure PFLZ.py exists in the 'PFL/Modules' folder.")

        print(f"✅ PFLZ.py file found: {pflz_file_path}")

        # Read the file directly as a Python module
        spec = importlib.util.spec_from_file_location("PFLZ_module_temp", pflz_file_path)
        if spec is None:
            raise ImportError(f"Unable to specify module for {pflz_file_path}.")

        module = importlib.util.module_from_spec(spec)
        sys.modules["PFLZ_module_temp"] = module
        spec.loader.exec_module(module)

        loaded_pflz_class = getattr(module, 'PFLZ', None)

        if loaded_pflz_class is None:
            raise ImportError("Class 'PFLZ' could not be found in the loaded module. Check the class name in PFLZ.py.")

        class PFLZ_CP_Alias(loaded_pflz_class):
            def __init__(self, *args, **kwargs):
                super().__init__(*args, **kwargs)
                if not hasattr(self, 'X_original_fit'):
                    self.X_original_fit = None

        return PFLZ_CP_Alias

# --- Module 4: Interpretability Calculations ---

class CalculateurInterpretabilite:
    """Calculates interpretability scores and generates clinical rules from the PFLZ model."""

    @staticmethod
    def calculer_features_interpretabilite(pflz_cp, X: np.ndarray, y: np.ndarray) -> Tuple[np.ndarray, Dict, Dict]:
        """
        Calculates interpretability features (scores, confidence, dominant clusters)
        and cluster interpretations (fuzzy rules).
        """
        logger.info("Calculating interpretability features ...")

        try:
            valeurs_appartenance = pflz_cp.calc_mf(X, normalized=True)
            if valeurs_appartenance is None or valeurs_appartenance.size == 0 or valeurs_appartenance.shape[1] == 0:
                raise ValueError("Invalid or empty fuzzy membership values after calc_mf.")
        except Exception as e:
            logger.error(f"Error calculating membership values: {e}")
            raise

        try:
            prob_conditionnelles = pflz_cp.calc_cond_probs(X, y)
            if prob_conditionnelles is None or prob_conditionnelles.size == 0 or prob_conditionnelles.shape[1] < 2:
                raise ValueError("Invalid or empty conditional probabilities after calc_cond_probs.")
        except Exception as e:
            logger.error(f"Error calculating conditional probabilities: {e}")
            raise

        n_samples = X.shape[0]
        scores_interpretabilite_bruts = np.zeros(n_samples)
        clusters_dominants = np.zeros(n_samples, dtype=int)
        scores_confiance = np.zeros(n_samples)

        num_clusters_actual = min(pflz_cp.n_clusters, valeurs_appartenance.shape[1])

        for i in range(n_samples):
            appartenance_i = valeurs_appartenance[i, :num_clusters_actual]
            prob_ponderees = appartenance_i * prob_conditionnelles[:num_clusters_actual, 1]

            if prob_ponderees.size > 0:
                max_prob_ponderee = np.max(prob_ponderees)
                scores_interpretabilite_bruts[i] = max_prob_ponderee
                clusters_dominants[i] = np.argmax(prob_ponderees)
            else:
                scores_interpretabilite_bruts[i] = 0.0
                clusters_dominants[i] = -1

            scores_confiance[i] = np.max(appartenance_i) if appartenance_i.size > 0 else 0.0

        if np.max(scores_interpretabilite_bruts) > np.min(scores_interpretabilite_bruts):
            scores_interpretabilite_normalises = (
                (scores_interpretabilite_bruts - np.min(scores_interpretabilite_bruts)) /
                (np.max(scores_interpretabilite_bruts) - np.min(scores_interpretabilite_bruts))
            )
        else:
            scores_interpretabilite_normalises = scores_interpretabilite_bruts

        seuil_adaptatif = np.percentile(scores_interpretabilite_normalises,
                                             ConfigurationSysteme.PERCENTILE_COVERAGE)

        diagnostics = CalculateurInterpretabilite._diagnostiquer_distributions(
            scores_interpretabilite_bruts,
            scores_interpretabilite_normalises,
            seuil_adaptatif
        )

        features_interpretabilite = np.column_stack([
            scores_interpretabilite_normalises,
            scores_confiance,
            clusters_dominants
        ])

        interpretations_clusters = {}
        if hasattr(pflz_cp, 'cluster_centers_') and pflz_cp.cluster_centers_.shape[0] > 0:
            rule_widths = pflz_cp.calc_rule_widths()
        else:
            logger.warning("PFLZ_CP centroids not available for rule width calculation. Using default widths.")
            num_features = X.shape[1] if X.shape[0] > 0 else 1
            rule_widths = np.full((pflz_cp.n_clusters, num_features), 0.5)

        alpha_rule = 1.5
        for cluster_id in range(num_clusters_actual):
            if hasattr(pflz_cp, 'cluster_centers_') and pflz_cp.cluster_centers_.shape[0] > cluster_id and pflz_cp.feature_names is not None:
                cluster_center = pflz_cp.cluster_centers_[cluster_id]
                regle_denormalisee = CalculateurInterpretabilite._generer_regle_centroide(
                    cluster_center,
                    pflz_cp.feature_names,
                    ConfigurationSysteme.SCALERS.get('min_max', {}),
                    alpha=alpha_rule,
                    widths=rule_widths[cluster_id] if rule_widths.shape[0] > cluster_id else np.full(cluster_center.shape[0], 0.5)
                )
                interpretations_clusters[cluster_id] = {
                    'prob_conditionnelle': prob_conditionnelles[cluster_id, 1],
                    'regle': regle_denormalisee
                }
            else:
                logger.warning(f"Cluster {cluster_id} or feature names not available. Skipping rule generation for this cluster.")

        logger.info(f"✅ Interpretability features calculated - Adaptive threshold: {seuil_adaptatif:.3f}")

        return features_interpretabilite, interpretations_clusters, diagnostics

    @staticmethod
    def _diagnostiquer_distributions(scores_bruts: np.ndarray, scores_normalises: np.ndarray,
                                     seuil_adaptatif: float) -> Dict:
        """Provides detailed diagnostics of interpretability score distributions."""
        diagnostics = {
            'scores_bruts_stats': {
                'min': np.min(scores_bruts), 'max': np.max(scores_bruts),
                'mean': np.mean(scores_bruts), 'std': np.std(scores_bruts),
                'percentiles': np.percentile(scores_bruts, [25, 50, 75, 90, 95])
            },
            'scores_normalises_stats': {
                'min': np.min(scores_normalises), 'max': np.max(scores_normalises),
                'mean': np.mean(scores_normalises), 'std': np.std(scores_normalises),
                'percentiles': np.percentile(scores_normalises, [25, 50, 75, 90, 95])
            },
            'seuil_adaptatif': seuil_adaptatif,
            'coverage_adaptatif': np.mean(scores_normalises > seuil_adaptatif)
        }

        diagnostics['scores_interpretabilite_normalises_original'] = scores_normalises
        logger.info(f"Diagnostic - Adaptive coverage: {diagnostics['coverage_adaptatif']:.1%}")
        return diagnostics

    @staticmethod
    def _generer_regle_centroide(centroide: np.ndarray, noms_features: List[str], scalers: Dict = None, alpha: float = 1.0, widths: Optional[np.ndarray] = None) -> str:
        """
        Generates a readable clinical rule from a centroid, with denormalization.
        Uses `noms_features` directly.
        Alpha is used to adjust interval widths.
        `widths` provides specific widths for each centroid feature.
        """
        if scalers is None:
            scalers = {}

        conditions = []
        num_features_to_process = min(len(centroide), len(noms_features))
        if len(centroide) != len(noms_features):
            logger.warning(f"Inconsistency: Centroid has {len(centroide)} elements but {len(noms_features)} feature names. Processing first {num_features_to_process}.")

        if widths is None or len(widths) != num_features_to_process:
            logger.warning(f"Widths not provided or incorrect dimension for _generer_regle_centroide. Using default width.")
            default_width = 0.5
            widths = np.full(num_features_to_process, default_width)

        for i in range(num_features_to_process):
            feature_name_original = noms_features[i]
            valeur_normalisee = centroide[i]
            feature_width = widths[i]

            lower_bound_norm = valeur_normalisee - alpha * feature_width
            upper_bound_norm = valeur_normalisee + alpha * feature_width

            if 'min_max' in scalers and feature_name_original in scalers['min_max']:
                original_min, original_max = scalers['min_max'][feature_name_original]
                lower_bound_denorm = lower_bound_norm * (original_max - original_min) + original_min
                upper_bound_denorm = upper_bound_norm * (original_max - original_min) + original_min
                conditions.append(f"{feature_name_original} ∈ [{lower_bound_denorm:.2f}, {upper_bound_denorm:.2f}]")
            else:
                conditions.append(f"{feature_name_original} ∈ [{lower_bound_norm:.3f}, {upper_bound_norm:.3f}]")

        regle_simplifiee = " AND ".join(conditions)

        return regle_simplifiee

# --- Module 5: Metrics with Calibration ---

class CalculateurMetriques:
    """Calculates performance metrics and manages post-training calibration."""

    @staticmethod
    def calculer_ece(y_true: np.ndarray, y_pred_proba: np.ndarray, n_bins: int = 10) -> float:
        """Calculates Expected Calibration Error (ECE)."""
        bin_edges = np.linspace(0, 1, n_bins + 1)
        bin_indices = np.digitize(y_pred_proba, bin_edges[:-1])
        ece = 0.0
        for i in range(1, n_bins + 1):
            in_bin = bin_indices == i
            bin_count = np.sum(in_bin)
            if bin_count > 0:
                bin_true_mean = np.mean(y_true[in_bin])
                bin_pred_mean = np.mean(y_pred_proba[in_bin])
                ece += np.abs(bin_true_mean - bin_pred_mean) * (bin_count / len(y_true))
        return ece

    @staticmethod
    def appliquer_calibration_post_entrainement(y_true: np.ndarray, y_pred_proba: np.ndarray,
                                                 method: str = 'isotonic') -> Tuple[np.ndarray, object]:
        """Applies post-training calibration to improve probability reliability."""
        logger.info(f"Applying {method} calibration...")
        if method == 'isotonic':
            calibrator = IsotonicRegression(out_of_bounds='clip')
        elif method == 'sigmoid':
            calibrator = LogisticRegression(solver='lbfgs')
        else:
            raise ValueError(f"Unknown calibration method: {method}")

        if method == 'sigmoid':
            logits = np.log(np.clip(y_pred_proba, 1e-15, 1-1e-15) /
                            np.clip(1-y_pred_proba, 1e-15, 1-1e-15))
            calibrator.fit(logits.reshape(-1, 1), y_true)
            y_pred_calibre = calibrator.predict_proba(logits.reshape(-1, 1))[:, 1]
        else:
            calibrator.fit(y_pred_proba, y_true)
            y_pred_calibre = calibrator.predict(y_pred_proba)

        logger.info("✅ Calibration applied")
        return y_pred_calibre, calibrator

# --- Module 6: Training with Calibration and Constraints ---

class EntraineurModele:
    """Trains the tree model with calibration and integrates interpretability features."""

    def __init__(self, config: ConfigurationSysteme):
        self.config = config
        self.config.valider_configuration()
        self.calibrators = {}

    def entrainer_et_evaluer(self, X_subset: pd.DataFrame, y_subset: np.ndarray,
                                 nom_subset: str, features_selectionnees: List[str]) -> Dict:
        """Trains and evaluates the tree model for a given subset."""
        logger.info(f"Starting processing for {nom_subset} with tree model.")
        start_time = time.time()

        global _PFLZ_MODULE_LOADED
        if not _PFLZ_MODULE_LOADED:
            raise RuntimeError("PFL module not loaded correctly. Cannot continue training.")

        try:
            pflz_cp = PFLZ_CP(
                n_clusters=self.config.NB_REGLES,
                epsilon=self.config.EPSILON,
                lgb_params=self.config.PARAMS_LGB,
                feature_names=features_selectionnees
            )
            pflz_cp.fit(X_subset.values, y_subset)
            pflz_cp.X_original_fit = X_subset.values

        except Exception as e:
            logger.error(f"Error during PFLZ_CP initialization or fitting for {nom_subset}: {e}")
            raise

        centroides = pflz_cp.cluster_centers_

        features_interp, interpretations_clusters, diagnostics = (
            CalculateurInterpretabilite.calculer_features_interpretabilite(
                pflz_cp, X_subset.values, y_subset
            )
        )

        self._afficher_diagnostics(diagnostics, nom_subset)
        self._afficher_interpretations_clusters(interpretations_clusters)

        results_train_eval = self._validation_croisee(
            X_subset.values, y_subset,
            pflz_cp, features_interp, nom_subset, diagnostics
        )

        self._afficher_resultats(results_train_eval, nom_subset)

        results_train_eval['temps_minutes'] = round((time.time() - start_time) / 60, 2)
        logger.info(f"✅ {nom_subset} processing completed in {results_train_eval['temps_minutes']} min for tree model.")

        return {
            'centroides': centroides,
            'predictions': results_train_eval['y_pred_proba_calibre'],
            'predictions_brutes': results_train_eval['y_pred_proba'],
            'resultats': results_train_eval,
            'pflz_cp': pflz_cp,
            'interpretations': interpretations_clusters,
            'diagnostics': diagnostics,
            'calibrator': self.calibrators.get(nom_subset)
        }

    def _validation_croisee(self, X: np.ndarray, y: np.ndarray, pflz_cp: Any,
                                 features_interp: np.ndarray, nom_subset: str,
                                 diagnostics: Dict) -> Dict:
        """Performs cross-validation of the tree model with calibration and feature weighting."""
        logger.info(f"Starting cross-validation for tree model ({nom_subset}).")

        skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

        metriques = {
            'accuracy': [], 'balanced_accuracy': [], 'kappa': [], 'mcc': [],
            'precision': [], 'recall': [], 'f1_score': [], 'specificity': [], 'pr_auc': [], 'roc_auc': [], 'log_loss': [], 'brier': [],
            'fdr': [], 'for': [], 'tn': [], 'fp': [], 'fn': [], 'tp': []
        }

        y_pred_proba_complet = np.zeros(len(y))
        y_pred_complet = np.zeros(len(y))
        ecarts_coherence_fold = []

        unique_classes, counts = np.unique(y, return_counts=True)
        if len(unique_classes) > 1:
            majority_class_count = counts[0] if unique_classes[0] == 0 else counts[1]
            minority_class_count = counts[1] if unique_classes[1] == 1 else counts[0]
            scale_pos_weight_val = majority_class_count / minority_class_count if minority_class_count > 0 else 1
        else:
            scale_pos_weight_val = 1

        for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
            logger.debug(f"Processing fold {fold+1}/5 for tree model.")

            X_train_fold = X[train_idx]
            X_val_fold = X[val_idx]
            y_train_fold = y[train_idx]
            y_val_fold = y[val_idx]

            fuzzy_train = pflz_cp.calc_mf(X_train_fold, normalized=True)
            fuzzy_val = pflz_cp.calc_mf(X_val_fold, normalized=True)

            interp_train = features_interp[train_idx]
            interp_val = features_interp[val_idx]

            poids_interpretabilite = 15.0

            interp_train_ponderees = interp_train * poids_interpretabilite
            interp_val_ponderees = interp_val * poids_interpretabilite

            if fuzzy_train.shape[1] == 0 and self.config.NB_REGLES > 0:
                fuzzy_train = np.zeros((X_train_fold.shape[0], self.config.NB_REGLES))
            if fuzzy_val.shape[1] == 0 and self.config.NB_REGLES > 0:
                fuzzy_val = np.zeros((X_val_fold.shape[0], self.config.NB_REGLES))

            if interp_train_ponderees.shape[1] == 0:
                interp_train_ponderees = np.zeros((X_train_fold.shape[0], 3))
            if interp_val_ponderees.shape[1] == 0:
                interp_val_ponderees = np.zeros((X_val_fold.shape[0], 3))

            X_train_etendu = np.hstack((X_train_fold, fuzzy_train, interp_train_ponderees))
            X_val_etendu = np.hstack((X_val_fold, fuzzy_val, interp_val_ponderees))

            lgb_params_fold = self.config.PARAMS_LGB.copy()
            lgb_params_fold['scale_pos_weight'] = scale_pos_weight_val

            train_data = lgb.Dataset(X_train_etendu, label=y_train_fold, free_raw_data=False)
            val_data = lgb.Dataset(X_val_etendu, label=y_val_fold, reference=train_data, free_raw_data=False)

            model = lgb.train(
                lgb_params_fold,
                train_data,
                num_boost_round=500,
                valid_sets=[val_data],
                callbacks=[
                    lgb.early_stopping(stopping_rounds=30, verbose=False),
                ]
            )

            y_pred_proba = model.predict(X_val_etendu)
            y_pred = (y_pred_proba > 0.2).astype(int)

            y_pred_proba_complet[val_idx] = y_pred_proba
            y_pred_complet[val_idx] = y_pred

            if interp_val.shape[1] > 0:
                ecart_fold = np.mean(np.abs(y_pred_proba - interp_val[:, 0]))
            else:
                ecart_fold = 1.0
            ecarts_coherence_fold.append(ecart_fold)

            self._calculer_metriques_fold(y_val_fold, y_pred, y_pred_proba, metriques)

            # Calculate TP, FP, TN, FN for the fold and add to lists
            try:
                cm = confusion_matrix(y_val_fold, y_pred, labels=[0, 1])
                tn, fp, fn, tp = cm.ravel()
            except ValueError:
                tn, fp, fn, tp = (0, 0, 0, 0) # Handle cases with a single class
                if len(np.unique(y_val_fold)) == 1:
                    if y_val_fold[0] == 0:
                        tn = np.sum(y_pred == 0)
                        fp = np.sum(y_pred == 1)
                    else:
                        tp = np.sum(y_pred == 1)
                        fn = np.sum(y_pred == 0)

            metriques['tn'].append(tn)
            metriques['fp'].append(fp)
            metriques['fn'].append(fn)
            metriques['tp'].append(tp)

        y_pred_proba_calibre, calibrator = CalculateurMetriques.appliquer_calibration_post_entrainement(
            y, y_pred_proba_complet, method=self.config.CALIBRATION_METHOD
        )
        self.calibrators[nom_subset] = calibrator

        ece_brut = CalculateurMetriques.calculer_ece(y, y_pred_proba_complet)
        ece_calibre = CalculateurMetriques.calculer_ece(y, y_pred_proba_calibre)

        return {
            'metriques': metriques,
            'y_pred_proba': y_pred_proba_complet,
            'y_pred_proba_calibre': y_pred_proba_calibre,
            'y_pred': y_pred_complet,
            'ece_brut': ece_brut,
            'ece_calibre': ece_calibre,
            'amelioration_ece': ece_brut - ece_calibre,
            'ecarts_coherence_folds': ecarts_coherence_fold,
            'ecart_coherence_moyen': np.mean(ecarts_coherence_fold)
        }

    def _calculer_metriques_fold(self, y_true: np.ndarray, y_pred: np.ndarray, y_pred_proba: np.ndarray, metriques: Dict):
        """Calculates performance metrics for a single fold, with handling of single class cases."""
        try:
            y_pred_proba = np.clip(y_pred_proba, 1e-15, 1 - 1e-15)

            if len(np.unique(y_true)) == 1 or len(np.unique(y_pred)) == 1:
                tn, fp, fn, tp = (0,0,0,0)
                if len(np.unique(y_true)) == 1 and y_true[0] == 0:
                    tn = np.sum(y_pred == 0)
                    fp = np.sum(y_pred == 1)
                elif len(np.unique(y_true)) == 1 and y_true[0] == 1:
                    tp = np.sum(y_pred == 1)
                    fn = np.sum(y_pred == 0)
                else:
                    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                    tn, fp, fn, tp = cm.ravel()
            else:
                cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
                tn, fp, fn, tp = cm.ravel()

            specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
            fdr = fp / (tp + fp) if (tp + fp) > 0 else 0
            for_rate = fn / (tn + fn) if (tn + fn) > 0 else 0

            metriques['accuracy'].append(accuracy_score(y_true, y_pred))
            metriques['balanced_accuracy'].append(balanced_accuracy_score(y_true, y_pred))
            metriques['kappa'].append(cohen_kappa_score(y_true, y_pred))
            metriques['mcc'].append(matthews_corrcoef(y_true, y_pred))
            metriques['precision'].append(precision_score(y_true, y_pred, zero_division=0))
            metriques['recall'].append(recall_score(y_true, y_pred, zero_division=0))
            metriques['f1_score'].append(f1_score(y_true, y_pred, zero_division=0))
            metriques['specificity'].append(specificity)

            if len(np.unique(y_true)) > 1:
                precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_proba)
                metriques['pr_auc'].append(auc(recall_vals, precision_vals))
                metriques['roc_auc'].append(roc_auc_score(y_true, y_pred_proba))
            else:
                metriques['pr_auc'].append(np.nan)
                metriques['roc_auc'].append(np.nan)

            metriques['log_loss'].append(log_loss(y_true, y_pred_proba))
            metriques['brier'].append(brier_score_loss(y_true, y_pred_proba))
            metriques['fdr'].append(fdr)
            metriques['for'].append(for_rate)

        except Exception as e:
            logger.error(f"Error calculating metrics for a fold: {e}. Filling with 0.0 or NaN.")
            for key in metriques.keys():
                if key in ['pr_auc', 'roc_auc']:
                    metriques[key].append(np.nan)
                else:
                    metriques[key].append(0.0)

    def _afficher_diagnostics(self, diagnostics: Dict, nom_subset: str):
        """Displays detailed interpretability diagnostics."""
        # print(f"\n🌟 INTERPRETABILITY DIAGNOSTICS FOR {nom_subset}")
        # print("-" * 60)
        # print(f"Raw scores - Min: {diagnostics['scores_bruts_stats']['min']:.4f}, "
        #      f"Max: {diagnostics['scores_bruts_stats']['max']:.4f}, "
        #      f"Mean: {diagnostics['scores_bruts_stats']['mean']:.4f}")
        # print(f"Normalized scores - Min: {diagnostics['scores_normalises_stats']['min']:.4f}, "
        #      f"Max: {diagnostics['scores_normalises_stats']['max']:.4f}, "
        #      f"Mean: {diagnostics['scores_normalises_stats']['mean']:.4f}")
        # print(f"Adaptive threshold ({ConfigurationSysteme.PERCENTILE_COVERAGE}th percentile): {diagnostics['seuil_adaptatif']:.4f}")
        # print(f"Adaptive coverage: {diagnostics['coverage_adaptatif']:.1%}")

    def _afficher_interpretations_clusters(self, interpretations_clusters: Dict):
        """Displays cluster interpretations (associated fuzzy rules)."""
        # print("\n🌟 Cluster-based interpretations:")
        for cluster_id, interpretation in interpretations_clusters.items():
            print(f"\nCluster {cluster_id}:")
            print(f"     • P(mortality|cluster) = {interpretation['prob_conditionnelle']:.2%}")
            # print(f"     • Rule: {interpretation['regle']}")

    def _afficher_resultats(self, resultats: Dict, nom_subset: str):
        """Displays performance and calibration metrics of the tree model."""
        print(f"\n{'-'*70}")
        print(f"METRICS FOR {nom_subset} (tree model)")
        print(f"{'-'*70}")

        metriques_moyennes = {
            'Accuracy': np.nanmean(resultats['metriques']['accuracy']) * 100,
            'Balanced Accuracy': np.nanmean(resultats['metriques']['balanced_accuracy']) * 100,
            'Kappa': np.nanmean(resultats['metriques']['kappa']),
            'MCC': np.nanmean(resultats['metriques']['mcc']),
            'Precision': np.nanmean(resultats['metriques']['precision']) * 100,
            'Recall': np.nanmean(resultats['metriques']['recall']) * 100,
            'F1-Score': np.nanmean(resultats['metriques']['f1_score']) * 100,
            'Specificity': np.nanmean(resultats['metriques']['specificity']) * 100,
            'PR-AUC': np.nanmean(resultats['metriques']['pr_auc']),
            'ROC-AUC': np.nanmean(resultats['metriques']['roc_auc']),
            'Log Loss': np.nanmean(resultats['metriques']['log_loss']),
            'Brier': np.nanmean(resultats['metriques'].get('brier', [np.nan])),
            'FDR': np.nanmean(resultats['metriques']['fdr']),
            'FOR': np.nanmean(resultats['metriques']['for'])
        }

        metriques_calibration = {
            'Raw ECE': resultats.get('ece_brut', np.nan),
            'Calibrated ECE': resultats.get('ece_calibre', np.nan),
            'ECE Improvement': resultats.get('amelioration_ece', np.nan),
        }

        print("🌟 TREE MODEL PERFORMANCE METRICS:")
        print("Accuracy                                     : {:.2f}%".format(metriques_moyennes['Accuracy']))
        print("Balanced Accuracy                            : {:.2f}%".format(metriques_moyennes['Balanced Accuracy']))
        print("Kappa                                        : {:.4f}".format(metriques_moyennes['Kappa']))
        print("MCC                                          : {:.4f}".format(metriques_moyennes['MCC']))
        print("Precision                                    : {:.2f}%".format(metriques_moyennes['Precision']))
        print("Recall                                       : {:.2f}%".format(metriques_moyennes['Recall']))
        print("F1-Score                                     : {:.2f}%".format(metriques_moyennes['F1-Score']))
        print("Specificity                                  : {:.2f}%".format(metriques_moyennes['Specificity']))
        print("PR-AUC                                       : {:.4f}".format(metriques_moyennes['PR-AUC'] if not np.isnan(metriques_moyennes['PR-AUC']) else 0.0))
        print("ROC-AUC                                      : {:.4f}".format(metriques_moyennes['ROC-AUC'] if not np.isnan(metriques_moyennes['ROC-AUC']) else 0.0))
        print("Log Loss                                     : {:.4f}".format(metriques_moyennes['Log Loss']))
        print("Brier Score                                  : {:.4f}".format(metriques_moyennes['Brier'] if not np.isnan(metriques_moyennes['Brier']) else 0.0))
        print("FDR                                          : {:.4f}".format(metriques_moyennes['FDR']))
        print("FOR                                          : {:.4f}".format(metriques_moyennes['FOR']))

        print("\n🌟 TREE MODEL CALIBRATION METRICS:")
        for nom, valeur in metriques_calibration.items():
            if 'ECE' in nom:
                print(f"     {nom:<20}: {valeur:.4f}")
            else:
                print(f"     {nom:<20}: {valeur:.4f}")

        print(f"{'-'*70}")

# --- Module 7: Risk Cluster Analyzer ---

class AnalyseurClustersRisque:
    """Risk cluster analysis with fixed clinical thresholds and consistency alerts."""

    @staticmethod
    def analyser_clusters_risque(X: pd.DataFrame, centroides: np.ndarray,
                                                 predictions: np.ndarray, nom_subset: str,
                                                 features_selectionnees: List[str],
                                                 pflz_cp: Any, interpretations_clusters: Dict,
                                                 predictions_brutes: Optional[np.ndarray] = None,
                                                 y_true: Optional[np.ndarray] = None,
                                                 interp_scores_normalises: Optional[np.ndarray] = None):
        """
        Performs detailed cluster analysis, classifying risk according to fixed clinical thresholds
        and evaluating consistency between the tree model and fuzzy rules.
        """
        logger.info(f"Risk cluster analysis for {nom_subset}")

        if centroides.shape[0] == 0 or X.empty or len(features_selectionnees) == 0:
            logger.warning(f"Centroids, X data or selected features empty for risk cluster analysis of {nom_subset}. Skipping detailed cluster analysis.")
            return {'alertes_coherence': []}

        valid_features_for_clustering = [f for f in features_selectionnees if f in X.columns]
        if not valid_features_for_clustering:
            logger.warning(f"None of the selected features ({features_selectionnees}) are present in DataFrame X. Cannot perform clustering for {nom_subset}. Skipping.")
            return {'alertes_coherence': []}

        km = KMeans(n_clusters=len(centroides), init=centroides, n_init=1, random_state=42)
        clusters = km.fit_predict(X[valid_features_for_clustering].values)

        print(f"\n{'-'*70}")
        print(f"RISK CLUSTERS FOR {nom_subset}")
        print(f"{'-'*70}")

        seuils = ConfigurationSysteme.SEUILS_RISQUE_CLINIQUES
        print("🌟 FIXED CLINICAL THRESHOLDS:")
        print(f"- LOW RISK: < {seuils['FAIBLE']:.1f}%")
        print(f"- MEDIUM RISK: {seuils['FAIBLE']:.1f}% to {seuils['MOYEN']:.1f}%")
        print(f"- HIGH RISK: ≥ {seuils['ELEVE']:.1f}%")
        print(f"{'-'*70}")

        risque_moyen_global = np.mean(predictions) * 100
        print(f"🌟 Global subset average risk: {risque_moyen_global:.1f}%")

        alertes_coherence_clusters = []

        for i in range(len(centroides)):
            masque_cluster = (clusters == i)
            n_patients = np.sum(masque_cluster)

            if n_patients == 0:
                print(f"\nCluster {i}: No patients")
                continue

            if i not in interpretations_clusters:
                logger.warning(f"Cluster ID {i} not found in 'interpretations_clusters'. Skipping detailed analysis for this cluster.")
                continue

            risque_moyen_modele_arborescent = np.mean(predictions[masque_cluster]) * 100

            if predictions_brutes is not None:
                risque_brut_modele_arborescent = np.mean(predictions_brutes[masque_cluster]) * 100
                amelioration_calibration = abs(risque_brut_modele_arborescent - risque_moyen_modele_arborescent)
            else:
                risque_brut_modele_arborescent = risque_moyen_modele_arborescent
                amelioration_calibration = 0

            if risque_moyen_modele_arborescent < seuils['FAIBLE']:
                niveau_risque = "LOW"
                issue_probable = "Probable survival"
                couleur_emoji = "🌟"
            elif risque_moyen_modele_arborescent <= seuils['MOYEN']:
                niveau_risque = "MEDIUM"
                issue_probable = "Enhanced monitoring"
                couleur_emoji = "✨"
            else:
                niveau_risque = "HIGH"
                issue_probable = "Urgent intervention"
                couleur_emoji = "🚨"

            prob_conditionnelle = interpretations_clusters[i].get('prob_conditionnelle', 0.0)
            regle_simplifiee = interpretations_clusters[i].get('regle', 'Rule not available')

            ecart_coherence_cluster = abs(risque_moyen_modele_arborescent/100 - prob_conditionnelle)

            alerte_coherence_specifique = ecart_coherence_cluster > ConfigurationSysteme.SEUIL_COHERENCE_ALERTE
            if alerte_coherence_specifique:
                alertes_coherence_clusters.append({
                    'cluster': i,
                    'ecart': ecart_coherence_cluster,
                    'risque_modele_arborescent': risque_moyen_modele_arborescent,
                    'prob_regles': prob_conditionnelle * 100
                })

            fidelity_cluster = 1 - ecart_coherence_cluster
            fidelity_cluster = max(0, min(1, fidelity_cluster))

            print(f"\n{couleur_emoji} Cluster {i} - {niveau_risque} RISK")
            print(f"     • Calibrated risk (tree model): {risque_moyen_modele_arborescent:.1f}%")
            if predictions_brutes is not None:
                print(f"     • Raw risk (tree model): {risque_brut_modele_arborescent:.1f}% (improvement: {amelioration_calibration:.1f}%)")
            print(f"     • P(mortality|cluster) (Fuzzy Rule): {prob_conditionnelle:.2%}")
            print(f"     • Cluster Fidelity                                 : {fidelity_cluster:.4f}")
            print(f"     • Clinical recommendation: {issue_probable}")
            print(f"     • Number of patients: {n_patients:,}")
            print(f"     • Percentage of subset: {(n_patients/len(predictions))*100:.1f}%")

            # print(f"     • Clinical rule: {regle_simplifiee}")

            status_coherence_display = ""
            if ecart_coherence_cluster < (ConfigurationSysteme.SEUIL_COHERENCE_ALERTE / 2):
                status_coherence_display = "✅ Excellent"
            elif ecart_coherence_cluster < ConfigurationSysteme.SEUIL_COHERENCE_ALERTE:
                status_coherence_display = "✨ Good"
            else:
                status_coherence_display = "🚨 Low - Further analysis required"

            print(f"     • Rule-model consistency: {status_coherence_display} (difference: {ecart_coherence_cluster:.1%})")

            if alerte_coherence_specifique:
                print(f"     🚨 ALERT: Significant difference detected - For this cluster, please consider:")
                print("       - Thorough review of feature interactions.")
                print("       - Validation with clinical expertise for these patient profiles.")

            print(f"{'-'*70}")

        if alertes_coherence_clusters:
            print(f"\n🚨 CONSISTENCY ALERTS SUMMARY BY CLUSTER ({len(alertes_coherence_clusters)} cluster(s)):")
            for alerte in alertes_coherence_clusters:
                print(f"     • Cluster {alerte['cluster']}: tree model={alerte['risque_modele_arborescent']:.1f}% vs Rules={alerte['prob_regles']:.1f}% (Difference: {alerte['ecart']:.1%})")
            print("\n✨ General Recommendations for Discrepancies:")
            print("     - **Rule Revision:** Collaborate with clinical expertise to refine or create new more precise rules for these patient profiles.")
            print("     - **Feature Enrichment:** Consider adding additional features to the tree model.")
        else:
            print(f"\n✅ SATISFACTORY CLUSTER CONSISTENCY: No significant differences detected.")

        # Visualizations are hidden
        # print("\n📈 RESULTS VISUALIZATIONS:")
        if y_true is not None and interp_scores_normalises is not None and len(predictions) > 0:
            plt.figure(figsize=(15, 6))
            plt.subplot(1, 2, 1)
            sns.histplot(predictions, color='skyblue', label='Tree model predictions (Calibrated)', kde=True, stat='density')
            sns.histplot(interp_scores_normalises, color='salmon', label='Interpretability scores (Rules)', kde=True, stat='density')
            plt.title(f'Score and Prediction Distribution for {nom_subset}')
            plt.xlabel('Probability / Score')
            plt.ylabel('Density')
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.6)
            plt.close()

            plt.subplot(1, 2, 2)
            bin_middles = np.linspace(0.05, 0.95, 10)
            fraction_of_positives = []
            mean_predicted_value = []
            if len(y_true) == len(predictions):
                for i in range(10):
                    low = i / 10.0
                    high = (i + 1) / 10.0
                    mask = (predictions >= low) & (predictions < high)
                    if np.any(mask):
                        fraction_of_positives.append(np.mean(y_true[mask]))
                        mean_predicted_value.append(np.mean(predictions[mask]))
                    else:
                        fraction_of_positives.append(np.nan)
                        mean_predicted_value.append(np.nan)
                plt.close()
            else:
                logger.warning(f"Cannot plot calibration curve for {nom_subset}: y_true and predictions have different lengths.")
        else:
            pass

        if alertes_coherence_clusters:
            cluster_ids = [a['cluster'] for a in alertes_coherence_clusters]
            ecarts_pourcentage = [a['ecart'] * 100 for a in alertes_coherence_clusters]
            plt.figure(figsize=(8, 5))
            sns.barplot(x=cluster_ids, y=ecarts_pourcentage, palette='viridis')
            plt.title(f'Tree Model vs Rules Consistency Difference by Cluster for {nom_subset}')
            plt.xlabel('Cluster ID')
            plt.ylabel('Consistency Difference (%)')
            plt.axhline(y=ConfigurationSysteme.SEUIL_COHERENCE_ALERTE * 100, color='r', linestyle='--', label='Alert Threshold')
            plt.legend()
            plt.grid(axis='y', linestyle='--', alpha=0.6)
            plt.tight_layout()
            plt.close()
        else:
            pass

        logger.info(f"✅ Fidelity and consistency measure analysis completed for {nom_subset}")
        return {
            'alertes_coherence': alertes_coherence_clusters
        }

# --- Module 8: Fidelity & Consistency Analyzer ---

class AnalyseurFideliteCohérence:
    """Calculates and presents global fidelity and consistency metrics of the model."""
    def __init__(self, config: ConfigurationSysteme):
        self.config = config

    def analyser_mesures_interpretabilite(self, y_pred_proba_modele_arborescent: np.ndarray,
                                             features_interp_scores: np.ndarray,
                                             y_true: np.ndarray,
                                             pflz_cp_instance: Any,
                                             nom_subset: str,
                                             interpretations_clusters_details: Dict):
        """
        Calculates and displays global fidelity and consistency metrics,
        as well as calibration diagnostics.
        """
        logger.info(f"Analyzing fidelity and consistency measures for {nom_subset}...")

        interp_scores_normalises = features_interp_scores[:, 0]

        ecart_moyen_global = np.mean(np.abs(y_pred_proba_modele_arborescent - interp_scores_normalises))
        coherence_globale_calc = 1 - ecart_moyen_global

        fidelity_calc = coherence_globale_calc

        seuil_adaptatif = np.percentile(interp_scores_normalises, self.config.PERCENTILE_COVERAGE)
        coverage_adaptatif_calc = np.mean(interp_scores_normalises >= seuil_adaptatif) * 100

        print(f"\n{'-'*70}")
        print(f"FIDELITY & CONSISTENCY SUMMARY FOR {nom_subset}")
        print(f"{'-'*70}")
        print(f"📊 Global Consistency (1 - Mean Absolute Difference): {coherence_globale_calc:.4f}")
        print(f"📊 Global Mean Absolute Difference: {ecart_moyen_global:.4f}")
        print(f"📊 Global Fidelity: {fidelity_calc:.4f}")
        print(f"📊 Adaptive Coverage (60th percentile): {coverage_adaptatif_calc:.2f}%")

        if ecart_moyen_global > self.config.SEUIL_COHERENCE_ALERTE:
            print("🚨 GLOBAL CONSISTENCY ALERT: The difference between model predictions and rule interpretations is significant.")
            print("    Further analysis of cluster discrepancies is recommended.")
        else:
            print("✅ SATISFACTORY GLOBAL CONSISTENCY: Model predictions are well aligned with rule interpretations.")

        X_df_for_plotting = pd.DataFrame()
        if hasattr(pflz_cp_instance, 'X_original_fit') and pflz_cp_instance.X_original_fit is not None and pflz_cp_instance.feature_names is not None:
            if pflz_cp_instance.X_original_fit.shape[1] == len(pflz_cp_instance.feature_names):
                X_df_for_plotting = pd.DataFrame(pflz_cp_instance.X_original_fit, columns=pflz_cp_instance.feature_names)
            else:
                logger.warning(f"Dimension inconsistency between X_original_fit ({pflz_cp_instance.X_original_fit.shape[1]} cols) and feature_names ({len(pflz_cp_instance.feature_names)} names). Skipping DataFrame creation for cluster analysis.")
        else:
            logger.warning("Original X data or feature names not available in PFLZ_CP instance. Skipping cluster visualizations.")

        if not X_df_for_plotting.empty and pflz_cp_instance.cluster_centers_.shape[0] > 0:
            AnalyseurClustersRisque.analyser_clusters_risque(
                X=X_df_for_plotting,
                centroides=pflz_cp_instance.cluster_centers_,
                predictions=y_pred_proba_modele_arborescent,
                nom_subset=nom_subset,
                features_selectionnees=pflz_cp_instance.feature_names,
                pflz_cp=pflz_cp_instance,
                interpretations_clusters=interpretations_clusters_details,
                y_true=y_true,
                interp_scores_normalises=interp_scores_normalises
            )
        else:
            print(f"\n(Cannot generate detailed cluster analysis for {nom_subset} because original X data or centroids are invalid.)")

        logger.info(f"✅ Fidelity and consistency measure analysis completed for {nom_subset}")

# --- Main Pipeline ---

class PipelinePrincipal:
    """Main pipeline for clinical integration, orchestrating all steps."""

    def __init__(self):
        self.config = ConfigurationSysteme()
        self.gestionnaire_donnees = GestionnaireDonnees()
        self.X_train = None
        self.y_train = None
        self.all_feature_names = None
        self.features_selectionnees = None

        self.config.valider_configuration()

        logger.info("🌟 Main pipeline initialized.")

    def _initialiser_et_charger_donnees(self):
        """Data initialization: mounts Drive, locates and loads datasets."""
        self.gestionnaire_donnees.monter_drive()
        self.gestionnaire_donnees.localiser_dossier_donnees()
        self.X_train, self.y_train, self.all_feature_names = self.gestionnaire_donnees.charger_donnees()

        if self.X_train is None or self.y_train is None:
            raise ValueError("Data loading failed")

        self.features_selectionnees = [col for col in self.X_train.columns if col != 'A']
        ConfigurationSysteme.NB_FEATURES = len(self.features_selectionnees)
        ConfigurationSysteme.INDICES_FEATURES = list(range(ConfigurationSysteme.NB_FEATURES))

        logger.info(f"✅ Data loaded: {self.X_train.shape[0]} patients, {len(self.features_selectionnees)} selected features (excluding 'A')")

        print("\n" + "="*50)
        print("SYSTEM CONFIGURATION")
        print("="*50)
        print("🏥 FIXED CLINICAL THRESHOLDS:")
        for niveau, seuil in self.config.SEUILS_RISQUE_CLINIQUES.items():
            print(f"     • {niveau} RISK: {seuil}%")
        # print("\n🔧 Features:")
        # print("     • 🌟 Adaptive thresholds for coverage (60th percentile)")
        # print("     • 🌟 Interpretability score normalization (0-1)")
        # print("     • 🌟 Post-training calibration (isotonic)")
        # print("     • 🌟 Alert system for inconsistencies (threshold: 15%)")
        # print("     • 🌟 Interpretability feature weighting (increased to 15)")
        # print("     • 🌟 Fixed clinical thresholds for comparability")
        # print("     • 🌟 Detailed logging and diagnostics")
        # print("     • 🌟 Class weighting to improve recall")
        # print("     • 🌟 Adjusted classification threshold to 0.2 to improve recall")
        # print("     • 🌟 Number of PFL rules: 7")
        # print("     • 🌟 All features (except 'A') used for rules")
        # print("\n📊 DEFINITIONS:")
        # print("     • y: Observed outcome - 1 (72h Mortality), 0 (Survival)")
        # print("     • A: Antibiotic treatment - 1 (Single), 0 (Multiple)")
        # print("     • **Accuracy**: Proportion of correct predictions (true positives + true negatives) relative to total observations.")
        # print("     • **Balanced Accuracy**: Average of recall (sensitivity) obtained on each class. Useful for imbalanced datasets.")
        # print("     • **Kappa (Cohen's Kappa)**: Measure of agreement between predictions and ground truth, correcting for chance agreement. A value of 1 indicates perfect agreement, 0 indicates random agreement.")
        # print("     • **MCC (Matthews Correlation Coefficient)**: Measure of quality of binary classifiers, even with imbalanced classes. Ranges from -1 (total disagreement) to +1 (perfect agreement), 0 being a random result.")
        # print("     • **Precision**: Proportion of true positives among all positive predictions. Answers: 'Among my positive predictions, how many are actually positive?'")
        # print("     • **Recall (Sensitivity)**: Proportion of true positives among all true positive cases (those that should have been predicted positive). Answers: 'Among all real positive cases, how many did I correctly identify?'")
        # print("     • **F1-Score**: Harmonic mean of precision and recall. A good balance between the two metrics, useful when seeking balance between false positives and false negatives.")
        # print("     • **Specificity**: Proportion of true negatives among all true negative cases. Measures the model's ability to correctly identify negative cases.")
        # print("     • **PR-AUC (Area Under the Precision-Recall Curve)**: Area under the precision-recall curve. Useful for evaluating models on highly imbalanced datasets, as it focuses on the performance of the positive minority class.")
        # print("     • **ROC-AUC (Area Under the Receiver Operating Characteristic Curve)**: Area under the ROC curve. Indicates the model's ability to distinguish classes. A value of 1 is perfect, 0.5 is random.")
        # print("     • **Log Loss (Logarithmic Loss)**: Measures the performance of a classification model where the prediction is a probability. Heavily penalizes wrong predictions made with high confidence.")
        # print("     • **Brier Score**: Measures the accuracy of probabilistic predictions. A value of 0 indicates perfect predictions, 1 the worst possible score.")
        # print("     • **FDR (False Discovery Rate)**: Proportion of false discoveries (false positives) among all predicted positive cases. FDR = FP / (FP + TP).")
        # print("     • **FOR (False Omission Rate)**: Proportion of false omissions (false negatives) among all predicted negative cases. FOR = FN / (FN + TN).")
        # print("     • **ECE (Expected Calibration Error)**: Measures the difference between the model's predicted confidence and actual accuracy. A low ECE indicates that the model's predicted probabilities are reliable.")
        # print("     • **Calibration Slope**: In a calibration plot, it's the slope of the regression line of observed proportions as a function of predicted probabilities. A slope of 1 indicates perfect calibration.")
        # print("     • **Consistency**: Degree of similarity between tree model predictions and interpretations based on fuzzy rules. A high value indicates good model interpretability through its associated rules.")
        # print("     • **Post-training calibration (isotonic)**: Technique to adjust a model's output probabilities to better reflect the true probabilities of the event. Isotonic regression is non-parametric and flexible, seeking a monotonic function to map raw probabilities to calibrated probabilities.")
        # print("     • **Adaptive coverage**: Percentage of patients whose predictions can be considered well interpreted by the rules, based on an adaptive interpretability score threshold (here, the 75th percentile). The higher this percentage, the more interpretable the model is for a large portion of cases.")
        print("="*50)
        # print(f"\n=== SELECTED FEATURES ({self.config.NB_FEATURES}) ===")
        # for i, nom in enumerate(self.features_selectionnees):
        #    print(f"     {i+1}. {nom}")

    def _charger_module_pflz(self):
        """Loads the PFLZ module from the specified directory."""
        global PFLZ_CP
        global _PFLZ_MODULE_LOADED
        try:
            loaded_pflz_class = ChargeurModulePFLZ.charger_module_pflz_cp(self.gestionnaire_donnees.mydrive_path)
            if loaded_pflz_class is None:
                raise ImportError("PFLZ class could not be found or loaded.")
            PFLZ_CP = loaded_pflz_class
            _PFLZ_MODULE_LOADED = True
            logger.info("✅ PFLZ_CP module loaded and validated")
        except Exception as e:
            logger.error(f"🚨 Error loading PFLZ_CP: {e}")
            _PFLZ_MODULE_LOADED = False
            raise


    def _preparer_subsets(self):
        """Prepares data subsets based on the 'A' variable."""
        logger.info("Preparing data subsets.")
        mask_A0 = self.X_train['A'] == 0
        mask_A1 = self.X_train['A'] == 1

        X_train_no_A = self.X_train.drop('A', axis=1)

        X_train_A0 = X_train_no_A[mask_A0]
        y_train_A0 = self.y_train[mask_A0]
        X_train_A1 = X_train_no_A[mask_A1]
        y_train_A1 = self.y_train[mask_A1]
        X_train_all = X_train_no_A
        y_train_all = self.y_train

        logger.info(f"Dataset A=0: {len(X_train_A0)} patients ({np.mean(y_train_A0):.1%} mortality)")
        logger.info(f"Dataset A=1: {len(X_train_A1)} patients ({np.mean(y_train_A1):.1%} mortality)")
        logger.info(f"Dataset ALL: {len(X_train_all)} patients ({np.mean(y_train_all):.1%} mortality)")
        return {
            'A0': (X_train_A0, y_train_A0),
            'A1': (X_train_A1, y_train_A1),
            'ALL': (X_train_all, y_train_all)
        }

    def _afficher_resume_final(self, results: Dict, analyses_clusters: Dict):
        """Displays a final summary of results and system functions."""
        print("\n" + "="*60)
        print("FINAL SUMMARY - PFL-TREE MODEL SYSTEM")
        print("="*60)

        # print("\n🌟 Features:")
        # print("     • ✨ Adaptive thresholds for Coverage (60th percentile)")
        # print("     • 🌟 Interpretability score normalization (0-1)")
        # print("     • ✨ Fixed clinical thresholds for inter-dataset comparability")
        # print("     • 🌟 Post-training calibration (ECE reduction)")
        # print("     • ✨ Alert system for inconsistencies (> 10% per cluster)")
        # print("     • 🌟 Increased interpretability feature weighting for tree model (15)")
        # print("     • ✨ Denormalized rules for enhanced clinical readability (alpha=1.5)")
        # print("     • 🌟 Detailed diagnostics and complete logging")
        # print("     • ✨ Class weighting to improve recall")
        # print("     • 🌟 Adjusted classification threshold to 0.2 to improve recall")
        # print("     • ✨ Number of PFL rules = 7")
        # print("     • 🌟 All features (except 'A') used for rules")

        print("\n✨ RESULTS BY DATASET:")
        for nom_subset in ["A=All", "A=0", "A=1"]:
            resultats_subset = results[nom_subset]['resultats']
            specific_analysis_results = analyses_clusters.get(nom_subset, {}).get('alertes_coherence', [])

            print(f"\n 🌟 {nom_subset}:")
            print(f"     • Tree model ROC-AUC: {np.nanmean(resultats_subset['metriques']['roc_auc'])*100:.2f}%")
            print(f"     • Raw ECE: {resultats_subset.get('ece_brut', np.nan):.4f}")
            print(f"     • Calibrated ECE: {resultats_subset.get('ece_calibre', np.nan):.4f}")
            print(f"     • ECE Improvement: {resultats_subset.get('amelioration_ece', np.nan):.4f}")
            print(f"     • Cluster inconsistency alerts: {len(specific_analysis_results)}")
            print(f"     • Recall: {np.nanmean(resultats_subset['metriques']['recall'])*100:.2f}%")

        # print("\n✨ CLINICAL IMPACT:")
        # print("     • Better calibrated probabilities → Increased confidence in tree model predictions.")
        # print("     • Fixed thresholds → Comparability between patient populations.")
        # print("     • Adaptive coverage → More patients with reliable interpretations (detailed by dedicated analysis).")
        # print("     • Cluster alert system → Targeted detection of complex cases for the tree model.")
        # print("     • Detailed diagnostics → Debugging and continuous improvement.")
        # print("     • Denormalized rules → Facilitated clinical decision making.")
        # print("     • Improved recall → Better detection of mortality cases.")

        # print("\n🌟 USAGE RECOMMENDATIONS:")
        # print("     1. ✨ Excellent Consistency (< 5% difference per cluster): Use rules as main guide, tree model confirms.")
        # print("     2. 🌟 Good Consistency (5-10% difference per cluster): Combine rules and tree model with caution, monitoring required.")
        # print("     3. ✨ Low Consistency (> 10% difference per cluster): Prioritize tree model predictions and investigate complex interactions.")
        # print("     4. 🌟 Regularly monitor model calibration with new patients.")
        # print("     5. ✨ Periodically recalibrate if ECE degradation is observed for the tree model.")

        print("\n" + "="*60)
        print("✨ SYSTEM READY FOR CLINICAL USE")
        print("="*60)
        logger.info("🌟 Final summary displayed successfully.")


    def executer(self):
        """Executes the entire pipeline: loading, training, evaluation and analysis."""
        try:
            logger.info("Starting pipeline execution.")

            self._initialiser_et_charger_donnees()
            self._charger_module_pflz()

            datasets = self._preparer_subsets()

            entraineur = EntraineurModele(self.config)
            analyseur_fidelite_coherence = AnalyseurFideliteCohérence(self.config)

            print("\n" + "="*50)
            print("TREE MODEL TRAINING AND EVALUATION")
            print("="*50)

            subsets_info = [
                ("A=0", "Multiple antibiotics", datasets['A0']),
                ("A=1", "Single antibiotic", datasets['A1']),
                ("A=All", "Combined dataset", datasets['ALL'])
            ]

            all_results = {}
            all_analysis_clusters = {}

            for nom_subset, description, (X_data_full, y_data) in subsets_info:
                print(f"\n🔧 Processing for {nom_subset} ({description})...")

                X_data_selected_features = X_data_full

                results_train_eval = entraineur.entrainer_et_evaluer(
                    X_data_selected_features, y_data, nom_subset, self.features_selectionnees
                )
                all_results[nom_subset] = results_train_eval

                print(f"\n✨ LAUNCHING DETAILED FIDELITY & CONSISTENCY ANALYSIS FOR {nom_subset}...")

                recalculated_features_interp, _, _ = CalculateurInterpretabilite.calculer_features_interpretabilite(
                    results_train_eval['pflz_cp'],
                    X_data_selected_features.values,
                    y_data
                )

                analyseur_fidelite_coherence.analyser_mesures_interpretabilite(
                    y_pred_proba_modele_arborescent=results_train_eval['predictions'],
                    features_interp_scores=recalculated_features_interp,
                    y_true=y_data,
                    pflz_cp_instance=results_train_eval['pflz_cp'],
                    nom_subset=nom_subset,
                    interpretations_clusters_details=results_train_eval['interpretations']
                )

                print(f"\n" + "="*50)
                print(f"RISK CLUSTER ANALYSIS FOR {nom_subset}")
                print(f"=")

                all_analysis_clusters[nom_subset] = AnalyseurClustersRisque.analyser_clusters_risque(
                    X_data_selected_features,
                    results_train_eval['centroides'],
                    results_train_eval['predictions'],
                    nom_subset,
                    self.features_selectionnees,
                    results_train_eval['pflz_cp'],
                    results_train_eval['interpretations'],
                    results_train_eval['predictions_brutes'],
                    y_true=y_data,
                    interp_scores_normalises=recalculated_features_interp[:, 0]
                )

                if nom_subset == "A=All":
                    logger.info("Calculating and storing global rule synthesis for 'A=All'...")

                    pflz_cp_instance_all = results_train_eval['pflz_cp']
                    interpretations_clusters_all = results_train_eval['interpretations']

                    if not hasattr(pflz_cp_instance_all, 'cluster_centers_') or pflz_cp_instance_all.cluster_centers_ is None or pflz_cp_instance_all.cluster_centers_.shape[0] == 0:
                        logger.warning("PFLZ centroids missing for 'A=All', cannot synthesize global rule.")
                    else:
                        cluster_centers_all = pflz_cp_instance_all.cluster_centers_
                        num_clusters_all = cluster_centers_all.shape[0]
                        num_features_all = cluster_centers_all.shape[1]

                        rule_widths_all = pflz_cp_instance_all.calc_rule_widths()
                        if rule_widths_all is None or rule_widths_all.shape[0] == 0:
                            logger.warning("Rule widths not available, using default.")
                            rule_widths_all = np.full((num_clusters_all, num_features_all), 0.5)

                        X_original_fit_all = pflz_cp_instance_all.X_original_fit
                        if X_original_fit_all is None or X_original_fit_all.shape[0] == 0:
                            logger.warning("X_original_fit missing for 'A=All', uniform cluster weighting.")
                            cluster_weights_all = np.ones(num_clusters_all) / num_clusters_all
                        else:
                            valeurs_appartenance_all = pflz_cp_instance_all.calc_mf(X_original_fit_all, normalized=True)
                            if valeurs_appartenance_all is None or valeurs_appartenance_all.shape[1] == 0:
                                cluster_weights_all = np.ones(num_clusters_all) / num_clusters_all
                            else:
                                cluster_weights_raw_all = np.sum(valeurs_appartenance_all, axis=0)
                                cluster_weights_all = cluster_weights_raw_all / np.sum(cluster_weights_raw_all)

                        meta_centroide_all = np.zeros(num_features_all)
                        global_mortality_prob_rules_all = 0.0
                        meta_widths_all = np.zeros(num_features_all)

                        for i in range(num_clusters_all):
                            weight = cluster_weights_all[i] if i < len(cluster_weights_all) else (1.0 / num_clusters_all)
                            meta_centroide_all += cluster_centers_all[i] * weight
                            global_mortality_prob_rules_all += interpretations_clusters_all[i].get('prob_conditionnelle', 0.0) * weight
                            if rule_widths_all.shape[0] > i:
                                meta_widths_all += rule_widths_all[i] * weight
                            else:
                                meta_widths_all += np.full(num_features_all, 0.5) * weight

                        if np.any(meta_widths_all <= 0):
                            meta_widths_all = np.full(num_features_all, 0.5)

                        regle_globale_formatee_all = CalculateurInterpretabilite._generer_regle_centroide(
                            centroide=meta_centroide_all,
                            noms_features=self.features_selectionnees,
                            scalers=self.config.SCALERS,
                            alpha=1.5,
                            widths=meta_widths_all
                        )

                        all_results[nom_subset]['synthese_globale_regle'] = {
                            'meta_centroide': meta_centroide_all,
                            'global_mortality_prob_rules': global_mortality_prob_rules_all,
                            'meta_widths': meta_widths_all,
                            'regle_formatee': regle_globale_formatee_all,
                        }
                        logger.info("✅ Global rule synthesis calculated and stored in results.")


            self._afficher_resume_final(all_results, all_analysis_clusters)

            logger.info("🌟 Pipeline executed successfully.")
            return all_results, all_analysis_clusters

        except Exception as e:
            logger.error(f"🚨 Error in pipeline: {str(e)}")
            traceback.print_exc()
            sys.exit(1)

# --- Main Entry Point ---

def afficher_regle_globale_finale_epuree(
    all_results_from_pipeline: Dict[str, Any],
    pipeline_config: Any
):
    """
    Displays a single global rule for the entire dataset,
    reading EXCLUSIVELY the already calculated and stored synthesized results from the pipeline.
    This function does NOT perform ANY heavy calculations or loading.
    """
    logger.info("Starting simplified display of unique global rule.")
    print("\n" + "="*70)
    print("✨ UNIQUE GLOBAL RULE FOR THE ENTIRE DATASET ✨")
    print("="*70)

    results_all_data = all_results_from_pipeline.get('A=All')

    if not results_all_data or 'synthese_globale_regle' not in results_all_data:
        logger.error("Global rule synthesis not available. Cannot display.")
        print("\n--- 🚨 ERROR: Global rule was not synthesized by the pipeline. Cannot display. ---")
        return

    synthese_globale_data = results_all_data['synthese_globale_regle']
    regle_globale_formatee = synthese_globale_data['regle_formatee']
    global_mortality_prob_rules = synthese_globale_data['global_mortality_prob_rules']

    predictions_calibrees_all = results_all_data['predictions']
    predictions_brutes_all = results_all_data['predictions_brutes']
    avg_pred_calib_model = np.mean(predictions_calibrees_all) if len(predictions_calibrees_all) > 0 else 0.0
    avg_pred_brute_model = np.mean(predictions_brutes_all) if len(predictions_brutes_all) > 0 else 0.0

    interp_scores_normalises_all = results_all_data['diagnostics'].get('scores_interpretabilite_normalises_original')

    if interp_scores_normalises_all is None or len(interp_scores_normalises_all) == 0:
        logger.error("Normalized interpretability scores missing for global consistency. Cannot calculate difference.")
        ecart_coherence_global = np.nan
        fidelity_globale = np.nan
    else:
        min_len = min(len(predictions_calibrees_all), len(interp_scores_normalises_all))
        if min_len == 0:
            ecart_coherence_global = np.nan
            fidelity_globale = np.nan
        else:
            ecart_coherence_global = np.mean(np.abs(predictions_calibrees_all[:min_len] - interp_scores_normalises_all[:min_len]))
            fidelity_globale = 1 - ecart_coherence_global


    nombre_patients_total = len(predictions_calibrees_all)

    print(f"• **Calibrated risk (tree model)**: {avg_pred_calib_model:.2%}")
    print(f"• **Raw risk (tree model)**: {avg_pred_brute_model:.2%}")
    print(f"• **P(mortality|Global Fuzzy Rule)**: {global_mortality_prob_rules:.2%}")
    print(f"• **Fidelity (tree model vs Global Rules)**: {fidelity_globale:.4f}")

    niveau_risque_global_rule = "N/A"
    if (global_mortality_prob_rules * 100) < pipeline_config.SEUILS_RISQUE_CLINIQUES['FAIBLE']:
        niveau_risque_global_rule = "LOW"
    elif (global_mortality_prob_rules * 100) <= pipeline_config.SEUILS_RISQUE_CLINIQUES['MOYEN']:
        niveau_risque_global_rule = "MEDIUM"
    else:
        niveau_risque_global_rule = "HIGH"
    print(f"• **Global clinical recommendation**: {niveau_risque_global_rule}")

    print(f"• **Number of patients in global dataset**: {nombre_patients_total:,}")
    print(f"• **Percentage of global subset**: 100.0%")
    # print(f"• **Global clinical rule**: **{regle_globale_formatee}**")

    status_coherence_display = ""
    if not np.isnan(ecart_coherence_global):
        if ecart_coherence_global < (pipeline_config.SEUIL_COHERENCE_ALERTE / 2):
            status_coherence_display = "✅ Excellent"
        elif ecart_coherence_global < pipeline_config.SEUIL_COHERENCE_ALERTE:
            status_coherence_display = "✨ Good"
        else:
            status_coherence_display = "🚨 Low - Further analysis required"
        print(f"• **Consistency (Rule vs tree model, Global)**: {status_coherence_display} (difference: {ecart_coherence_global:.1%})")

        if ecart_coherence_global > pipeline_config.SEUIL_COHERENCE_ALERTE:
            print("\n     🚨 GLOBAL ALERT: A significant difference between model predictions and the global rule has been detected.")
            print("       This indicates that the 'average profile' of the rule does not fully capture the complexity of the prediction model.")
            print("       A review of specific cluster alerts (displayed previously) is recommended to target problematic subgroups.")
    else:
        print(f"• **Consistency (Rule vs tree model, Global)**: Not calculable (missing or inconsistent data)")


    print("\n" + "="*70)
    print("✨ GLOBAL RULE SYNTHESIZED AND DISPLAYED SUCCESSFULLY")
    print("="*70)

    logger.info("Unique global rule synthesized and displayed successfully.")


if __name__ == "__main__":
    try:
        warnings.filterwarnings('ignore')
        logger.info("🌟 Starting PFL system.")

        pipeline = PipelinePrincipal()
        results, analyses = pipeline.executer()

        logger.info("✨ Pipeline execution completed successfully!")

        # Metrics collection
        metriques_A0 = results['A=0']['resultats']['metriques']
        tn_A0 = np.sum(metriques_A0['tn'])
        fp_A0 = np.sum(metriques_A0['fp'])
        fn_A0 = np.sum(metriques_A0['fn'])
        tp_A0 = np.sum(metriques_A0['tp'])

        metriques_A1 = results['A=1']['resultats']['metriques']
        tn_A1 = np.sum(metriques_A1['tn'])
        fp_A1 = np.sum(metriques_A1['fp'])
        fn_A1 = np.sum(metriques_A1['fn'])
        tp_A1 = np.sum(metriques_A1['tp'])

        metriques_ALL = results['A=All']['resultats']['metriques']
        tn_ALL = np.sum(metriques_ALL['tn'])
        fp_ALL = np.sum(metriques_ALL['fp'])
        fn_ALL = np.sum(metriques_ALL['fn'])
        tp_ALL = np.sum(metriques_ALL['tp'])

        print("\n" + "="*70)
        print("SUMMARY TABLE OF TRUE/FALSE POSITIVES AND NEGATIVES")
        print("="*70)

        data = {
            'Dataset': ['A=0', 'A=1', 'A=All'],
            'True Positives (TP)': [tp_A0, tp_A1, tp_ALL],
            'False Positives (FP)': [fp_A0, fp_A1, fp_ALL],
            'True Negatives (TN)': [tn_A0, tn_A1, tn_ALL],
            'False Negatives (FN)': [fn_A0, fn_A1, fn_ALL]
        }

        df_results = pd.DataFrame(data)
        print(df_results.to_string(index=False))

        print("\n" + "="*70)
        print("COMPARATIVE TABLE OF MAIN METRICS")
        print("="*70)

        comparaison = {}
        for subset in ['A=0', 'A=1', 'A=All']:
            results_subset = results[subset]['resultats']
            metriques_moyennes = {
                'Accuracy': np.nanmean(results_subset['metriques']['accuracy']),
                'Balanced Accuracy': np.nanmean(results_subset['metriques']['balanced_accuracy']),
                'ROC-AUC': np.nanmean(results_subset['metriques']['roc_auc']),
                'Recall': np.nanmean(results_subset['metriques']['recall']),
                'Precision': np.nanmean(results_subset['metriques']['precision']),
                'F1-Score': np.nanmean(results_subset['metriques']['f1_score']),
                'Calibrated ECE': results_subset.get('ece_calibre', np.nan),
            }
            comparaison[subset] = metriques_moyennes

        df_comparaison = pd.DataFrame(comparaison).T
        print(df_comparaison.to_string(float_format="%.4f"))


        afficher_regle_globale_finale_epuree(
            all_results_from_pipeline=results,
            pipeline_config=pipeline.config
        )

        logger.info("🌟 Complete process finished.")

    except Exception as e:
        logger.error(f"🚨 Fatal error: {e}")
        traceback.print_exc()
        sys.exit(1)

DRIVE MOUNTING
Mounted at /content/drive
Drive mounted successfully

DATA FOLDER LOCATION
✅ modeling_data2 folder found: /content/drive/MyDrive/PFL/modeling_data2

DATA LOADING

🌟 Data loaded successfully!

SYSTEM CONFIGURATION
🏥 FIXED CLINICAL THRESHOLDS:
     • FAIBLE RISK: 10.0%
     • MOYEN RISK: 15.0%
     • ELEVE RISK: 25.0%

PFL MODULE LOADING
✅ PFLZ.py file found: /content/drive/MyDrive/PFL/Modules/PFLZ.py

TREE MODEL TRAINING AND EVALUATION

🔧 Processing for A=0 (Multiple antibiotics)...



Cluster 0:
     • P(mortality|cluster) = 6.64%

Cluster 1:
     • P(mortality|cluster) = 12.30%

Cluster 2:
     • P(mortality|cluster) = 18.96%

Cluster 3:
     • P(mortality|cluster) = 4.05%

Cluster 4:
     • P(mortality|cluster) = 4.83%

Cluster 5:
     • P(mortality|cluster) = 10.15%

Cluster 6:
     • P(mortality|cluster) = 25.19%

----------------------------------------------------------------------
METRICS FOR A=0 (tree model)
----------------------------------------------------------------------
🌟 TREE MODEL PERFORMANCE METRICS:
Accuracy                                     : 89.04%
Balanced Accuracy                            : 71.33%
Kappa                                        : 0.3918
MCC                                          : 0.3949
Precision                                    : 41.83%
Recall                                       : 49.68%
F1-Score                                     : 45.18%
Specificity                                  : 92.97%
PR-AUC                


----------------------------------------------------------------------
FIDELITY & CONSISTENCY SUMMARY FOR A=0
----------------------------------------------------------------------
📊 Global Consistency (1 - Mean Absolute Difference): 0.7797
📊 Global Mean Absolute Difference: 0.2203
📊 Global Fidelity: 0.7797
📊 Adaptive Coverage (60th percentile): 40.00%
🚨 GLOBAL CONSISTENCY ALERT: The difference between model predictions and rule interpretations is significant.
    Further analysis of cluster discrepancies is recommended.

----------------------------------------------------------------------
RISK CLUSTERS FOR A=0
----------------------------------------------------------------------
🌟 FIXED CLINICAL THRESHOLDS:
- LOW RISK: < 10.0%
- MEDIUM RISK: 10.0% to 15.0%
- HIGH RISK: ≥ 25.0%
----------------------------------------------------------------------
🌟 Global subset average risk: 9.1%

🌟 Cluster 0 - LOW RISK
     • Calibrated risk (tree model): 9.7%
     • P(mortality|cluster) (Fuzzy 


Cluster 0:
     • P(mortality|cluster) = 5.60%

Cluster 1:
     • P(mortality|cluster) = 7.87%

Cluster 2:
     • P(mortality|cluster) = 49.56%

Cluster 3:
     • P(mortality|cluster) = 7.78%

Cluster 4:
     • P(mortality|cluster) = 8.02%

Cluster 5:
     • P(mortality|cluster) = 13.27%

Cluster 6:
     • P(mortality|cluster) = 22.55%

----------------------------------------------------------------------
METRICS FOR A=1 (tree model)
----------------------------------------------------------------------
🌟 TREE MODEL PERFORMANCE METRICS:
Accuracy                                     : 73.95%
Balanced Accuracy                            : 77.81%
Kappa                                        : 0.2888
MCC                                          : 0.3648
Precision                                    : 26.66%
Recall                                       : 82.73%
F1-Score                                     : 40.32%
Specificity                                  : 72.90%
PR-AUC                 


----------------------------------------------------------------------
FIDELITY & CONSISTENCY SUMMARY FOR A=1
----------------------------------------------------------------------
📊 Global Consistency (1 - Mean Absolute Difference): 0.8993
📊 Global Mean Absolute Difference: 0.1007
📊 Global Fidelity: 0.8993
📊 Adaptive Coverage (60th percentile): 40.00%
🚨 GLOBAL CONSISTENCY ALERT: The difference between model predictions and rule interpretations is significant.
    Further analysis of cluster discrepancies is recommended.

----------------------------------------------------------------------
RISK CLUSTERS FOR A=1
----------------------------------------------------------------------
🌟 FIXED CLINICAL THRESHOLDS:
- LOW RISK: < 10.0%
- MEDIUM RISK: 10.0% to 15.0%
- HIGH RISK: ≥ 25.0%
----------------------------------------------------------------------
🌟 Global subset average risk: 10.6%

🌟 Cluster 0 - LOW RISK
     • Calibrated risk (tree model): 5.4%
     • P(mortality|cluster) (Fuzzy


Cluster 0:
     • P(mortality|cluster) = 11.45%

Cluster 1:
     • P(mortality|cluster) = 11.88%

Cluster 2:
     • P(mortality|cluster) = 5.06%

Cluster 3:
     • P(mortality|cluster) = 12.26%

Cluster 4:
     • P(mortality|cluster) = 42.13%

Cluster 5:
     • P(mortality|cluster) = 4.30%

Cluster 6:
     • P(mortality|cluster) = 7.84%

----------------------------------------------------------------------
METRICS FOR A=All (tree model)
----------------------------------------------------------------------
🌟 TREE MODEL PERFORMANCE METRICS:
Accuracy                                     : 87.86%
Balanced Accuracy                            : 69.78%
Kappa                                        : 0.3597
MCC                                          : 0.3621
Precision                                    : 38.82%
Recall                                       : 47.45%
F1-Score                                     : 42.68%
Specificity                                  : 92.11%
PR-AUC              


----------------------------------------------------------------------
FIDELITY & CONSISTENCY SUMMARY FOR A=All
----------------------------------------------------------------------
📊 Global Consistency (1 - Mean Absolute Difference): 0.8806
📊 Global Mean Absolute Difference: 0.1194
📊 Global Fidelity: 0.8806
📊 Adaptive Coverage (60th percentile): 40.00%
🚨 GLOBAL CONSISTENCY ALERT: The difference between model predictions and rule interpretations is significant.
    Further analysis of cluster discrepancies is recommended.

----------------------------------------------------------------------
RISK CLUSTERS FOR A=All
----------------------------------------------------------------------
🌟 FIXED CLINICAL THRESHOLDS:
- LOW RISK: < 10.0%
- MEDIUM RISK: 10.0% to 15.0%
- HIGH RISK: ≥ 25.0%
----------------------------------------------------------------------
🌟 Global subset average risk: 9.5%

✨ Cluster 0 - MEDIUM RISK
     • Calibrated risk (tree model): 13.0%
     • P(mortality|cluster)


FINAL SUMMARY - PFL-TREE MODEL SYSTEM

✨ RESULTS BY DATASET:

 🌟 A=All:
     • Tree model ROC-AUC: 81.82%
     • Raw ECE: 0.0893
     • Calibrated ECE: 0.0000
     • ECE Improvement: 0.0893
     • Cluster inconsistency alerts: 0
     • Recall: 47.45%

 🌟 A=0:
     • Tree model ROC-AUC: 83.07%
     • Raw ECE: 0.0917
     • Calibrated ECE: 0.0000
     • ECE Improvement: 0.0917
     • Cluster inconsistency alerts: 0
     • Recall: 49.68%

 🌟 A=1:
     • Tree model ROC-AUC: 85.68%
     • Raw ECE: 0.0817
     • Calibrated ECE: 0.0000
     • ECE Improvement: 0.0817
     • Cluster inconsistency alerts: 1
     • Recall: 82.73%

✨ SYSTEM READY FOR CLINICAL USE

SUMMARY TABLE OF TRUE/FALSE POSITIVES AND NEGATIVES
Dataset  True Positives (TP)  False Positives (FP)  True Negatives (TN)  False Negatives (FN)
    A=0                21762                 30809               407656                 22042
    A=1                16691                 45948               123615                  3485
  A=